In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import initial_conditions
import constants
import misc
import state
import residual
import jacobian

In [ ]:
r_first = constants.FLOATDTYPE(0.001)
r_outer = constants.FLOATDTYPE(200.0)
n_shell = 200

In [ ]:
ln_r_edge, u, dm, m_edge = initial_conditions.set_up_initial_conditions(
    r_first, r_outer, n_shell
)

In [ ]:
sigma_over_m = constants.FLOATDTYPE(0.3)
beta = constants.FLOATDTYPE(0.8)
alpha = constants.FLOATDTYPE(1.0)

In [ ]:
state_init = state.state_from_unknowns(ln_r_edge, u, dm, sigma_over_m, beta, alpha)

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(15, 5), sharex="all")
ax[0].plot(state_init["ln_r"], state_init["ln_rho"])
ax[1].plot(state_init["ln_r"], dm)
ax[2].plot(ln_r_edge, m_edge)
ax[3].plot(state_init["ln_r"], state_init["u"])

ax[1].set_yscale("log")
ax[2].set_yscale("log")

In [ ]:
def residual_for_numerical_jacobian(
    x, v_old, u_old, m_edge, dm, sigma_over_m, beta, alpha, dt, r_outer, n_shell
):
    ln_r_edge, u = misc.unpack_unknowns(x, r_outer, n_shell)
    state_curr = state.state_from_unknowns(ln_r_edge, u, dm, sigma_over_m, beta, alpha)
    return residual.residual(state_curr, v_old, u_old, m_edge, dm, dt)

In [ ]:
def numerical_jacobian(F, x, step, *args):

    J = np.empty((len(x), len(x)), dtype=constants.FLOATDTYPE)

    for j in range(len(x)):
        h = step

        x_plus = x.copy()
        x_minus = x.copy()

        x_plus[j] += h
        x_minus[j] -= h

        J[:, j] = (F(x_plus, *args) - F(x_minus, *args)) / (2.0 * h)

    return J

In [ ]:
v_old = state_init["v"]
u_old = state_init["u"]
dt = 0.001

point = np.concatenate((ln_r_edge[1:-1], u))
jac_num = numerical_jacobian(
    residual_for_numerical_jacobian,
    point,
    1e-6,
    v_old,
    u_old,
    m_edge,
    dm,
    sigma_over_m,
    beta,
    alpha,
    dt,
    r_outer,
    n_shell,
)

jac = jacobian.analytic_jacobian(state_init, v_old, m_edge, dm, dt, alpha)

In [ ]:
a = 0
b = n_shell - 1
c = 0
d = n_shell - 1
rel_err = np.abs(jac[a:b, c:d] - jac_num[a:b, c:d]) / np.maximum(
    1.0, np.maximum(np.abs(jac[a:b, c:d]), np.abs(jac_num[a:b, c:d]))
)
np.max(rel_err), np.max(np.abs(jac[a:b, c:d] - jac_num[a:b, c:d]))

In [ ]:
a = 0
b = n_shell - 1
c = n_shell - 1
d = 2 * n_shell
rel_err = np.abs(jac[a:b, c:d] - jac_num[a:b, c:d]) / np.maximum(
    1.0, np.maximum(np.abs(jac[a:b, c:d]), np.abs(jac_num[a:b, c:d]))
)
np.max(rel_err), np.max(np.abs(jac[a:b, c:d] - jac_num[a:b, c:d]))

In [ ]:
a = n_shell - 1
b = 2 * n_shell
c = 0
d = n_shell - 1
rel_err = np.abs(jac[a:b, c:d] - jac_num[a:b, c:d]) / np.maximum(
    1.0, np.maximum(np.abs(jac[a:b, c:d]), np.abs(jac_num[a:b, c:d]))
)
np.max(rel_err), np.max(np.abs(jac[a:b, c:d] - jac_num[a:b, c:d]))

In [ ]:
a = n_shell - 1
b = 2 * n_shell
c = n_shell - 1
d = 2 * n_shell
rel_err = np.abs(jac[a:b, c:d] - jac_num[a:b, c:d]) / np.maximum(
    1.0, np.maximum(np.abs(jac[a:b, c:d]), np.abs(jac_num[a:b, c:d]))
)
np.max(rel_err), np.max(np.abs(jac[a:b, c:d] - jac_num[a:b, c:d]))